In [2]:
import pandas as pd
from pyarrow import flight


In [3]:
# --- Configuration ---
# Dremio Arrow Flight port is 32010 (as defined in your docker-compose)
FLIGHT_URL = "grpc://localhost:32010" 
USERNAME = "laimeng"      # Your Dremio user
PASSWORD = "password123"  # Your Dremio password
CATALOG = "Nessie_catalog"

In [5]:
# 1. Connect to the Dremio Flight Server
client = flight.FlightClient(FLIGHT_URL)

# 2. Authenticate
# Note: Flight uses Basic Auth under the hood
token = client.authenticate_basic_token(USERNAME, PASSWORD)
options = flight.FlightCallOptions(headers=[token])

In [ ]:
# 3. Define the query
# Let's query one of the Iceberg tables you just created!
table_name = "balance_sheet_iceberg"
query = f'SELECT * FROM "{CATALOG}"."{table_name}" LIMIT 10'

print(f"Executing Query: {query}\n")

In [8]:
# 4. Execute the query and fetch the results
# get_flight_info tells the server to prepare the query execution
flight_info = client.get_flight_info(flight.FlightDescriptor.for_command(query), options)

# We grab the first (and usually only) endpoint to stream the data from
endpoint = flight_info.endpoints[0]
reader = client.do_get(endpoint.ticket, options)

# 5. Read the data directly into a PyArrow Table...
arrow_table = reader.read_all()

# 6. ...and convert it to a Pandas DataFrame for easy viewing/manipulation!
df = arrow_table.to_pandas()

# Display the results
print("Results:")
print(df)

Results:
             A                            B           C
0     Category                         Item      Amount
1       Assets    Cash and Cash Equivalents  2709100.39
2       Assets          Accounts Receivable   526062.56
3       Assets                    Inventory   395448.23
4       Assets         Total Current Assets  3630611.18
5       Assets  Property, Plant & Equipment    850000.0
6       Assets            Intangible Assets    250000.0
7       Assets                 Total Assets  4730611.18
8  Liabilities             Accounts Payable   263632.15
9  Liabilities              Short Term Debt    150000.0


In [11]:
reader

In [10]:
endpoint

<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'\n?SELECT * FROM "Nessie_catalog"."balance_sheet_iceberg" LIMIT 10\x12Y\nW\n?SELECT * FROM "Nessie_catalog"."balance_sheet_iceberg" LIMIT 10\x10\x02\x1a\x12\tV\xb4\x89\x1f\xd9\'\xfa\x15\x11\x00\xacX\xc58L\x86T'> locations=[] expiration_time=None app_metadata=b''>

In [9]:
flight_info

<pyarrow.flight.FlightInfo schema=A: string
  -- field metadata --
  ARROW:FLIGHT:SQL:PRECISION: '65536'
  ARROW:FLIGHT:SQL:IS_AUTO_INCREMENT: '0'
  ARROW:FLIGHT:SQL:IS_CASE_SENSITIVE: '0'
  ARROW:FLIGHT:SQL:SCHEMA_NAME: ''
  ARROW:FLIGHT:SQL:TABLE_NAME: ''
  ARROW:FLIGHT:SQL:IS_SEARCHABLE: '1'
  ARROW:FLIGHT:SQL:IS_READ_ONLY: '1'
  ARROW:FLIGHT:SQL:TYPE_NAME: 'CHARACTER VARYING'
B: string
  -- field metadata --
  ARROW:FLIGHT:SQL:PRECISION: '65536'
  ARROW:FLIGHT:SQL:IS_AUTO_INCREMENT: '0'
  ARROW:FLIGHT:SQL:IS_CASE_SENSITIVE: '0'
  ARROW:FLIGHT:SQL:SCHEMA_NAME: ''
  ARROW:FLIGHT:SQL:TABLE_NAME: ''
  ARROW:FLIGHT:SQL:IS_SEARCHABLE: '1'
  ARROW:FLIGHT:SQL:IS_READ_ONLY: '1'
  ARROW:FLIGHT:SQL:TYPE_NAME: 'CHARACTER VARYING'
C: string
  -- field metadata --
  ARROW:FLIGHT:SQL:PRECISION: '65536'
  ARROW:FLIGHT:SQL:IS_AUTO_INCREMENT: '0'
  ARROW:FLIGHT:SQL:IS_CASE_SENSITIVE: '0'
  ARROW:FLIGHT:SQL:SCHEMA_NAME: ''
  ARROW:FLIGHT:SQL:TABLE_NAME: ''
  ARROW:FLIGHT:SQL:IS_SEARCHABLE: '1'
  ARRO